# Kaggle Training Notebook — AI-Durian-Disease-Detection

Hướng dẫn sử dụng:
1. Upload project và dataset lên Kaggle
2. Import notebook này vào Kaggle
3. Chỉnh sửa **Cell 2** — thay đổi đường dẫn
4. Chỉnh sửa **Cell 4** — chọn model muốn train
5. Chạy **Runtime > Run all**

Thứ tự train: ResNet18 > MobileNetV3 > EfficientNet-B0 > ResNet34 > ResNet50

In [1]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN
!pip install albumentations PyYAML tensorboard scikit-learn matplotlib seaborn opencv-python tqdm pillow pandas -q

In [2]:
# BƯỚC 2: KHAI BÁO ĐƯỜNG DẪN + DEBUG
import os
import sys
from pathlib import Path
from PIL import Image

# ============================================================
# THAY ĐỔI ĐƯỜNG DẪN PHÙ HỢP VỚI DATASET CỦA BẠN
# ============================================================
PROJECT_ROOT = Path("/kaggle/input/datasets/tranmanh1312/ai-durian-disease-detection")
# data-leaf chứa trực tiếp train/val/test (KHÔNG có thư mục con data/processed)
DATA_ROOT = Path("/kaggle/input/datasets/tranmanh1312/durian-processed-data/data-leaf")
# ============================================================

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT} (exists: {PROJECT_ROOT.exists()})")
print(f"Data root: {DATA_ROOT} (exists: {DATA_ROOT.exists()})")

# --- DEBUG: Xem cấu trúc thư mục ---
print("\n--- DEBUG: Cấu trúc thư mục ---")
if DATA_ROOT.exists():
    for item in sorted(DATA_ROOT.iterdir()):
        if item.is_dir():
            print(f"  [DIR]  {item.name}/")
            for sub in sorted(item.iterdir())[:5]:
                if sub.is_dir():
                    count = sum(1 for f in sub.rglob("*.*") if f.is_file())
                    print(f"    [DIR]  {sub.name}/ ({count} files)")
                else:
                    print(f"    [FILE] {sub.name}")
        else:
            print(f"  [FILE] {item.name}")
else:
    print("  DATA_ROOT không tồn tại!")

# Extensions để đếm ảnh
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

train_path = DATA_ROOT / "train"
val_path = DATA_ROOT / "val"
test_path = DATA_ROOT / "test"

def count_images(folder):
    if not folder.exists():
        return 0
    count = 0
    for ext in IMG_EXTENSIONS:
        count += sum(1 for _ in folder.rglob(f"*{ext}"))
    return count

print(f"\nTrain: {train_path} ({count_images(train_path)} images)")
print(f"Val:   {val_path} ({count_images(val_path)} images)")
print(f"Test:  {test_path} ({count_images(test_path)} images)")

Project root: /kaggle/input/datasets/tranmanh1312/ai-durian-disease-detection (exists: True)
Data root: /kaggle/input/datasets/tranmanh1312/durian-processed-data/data-leaf (exists: True)

--- DEBUG: Cấu trúc thư mục ---
  [DIR]  test/
    [DIR]  ALGAL_LEAF_SPOT/ (147 files)
    [DIR]  ALLOCARIDARA_ATTACK/ (183 files)
    [DIR]  HEALTHY_LEAF/ (196 files)
    [DIR]  LEAF_BLIGHT/ (188 files)
    [DIR]  PHOMOPSIS_LEAF_SPOT/ (176 files)
  [DIR]  train/
    [DIR]  ALGAL_LEAF_SPOT/ (513 files)
    [DIR]  ALLOCARIDARA_ATTACK/ (639 files)
    [DIR]  HEALTHY_LEAF/ (683 files)
    [DIR]  LEAF_BLIGHT/ (655 files)
    [DIR]  PHOMOPSIS_LEAF_SPOT/ (614 files)
  [DIR]  val/
    [DIR]  ALGAL_LEAF_SPOT/ (73 files)
    [DIR]  ALLOCARIDARA_ATTACK/ (91 files)
    [DIR]  HEALTHY_LEAF/ (97 files)
    [DIR]  LEAF_BLIGHT/ (94 files)
    [DIR]  PHOMOPSIS_LEAF_SPOT/ (88 files)

Train: /kaggle/input/datasets/tranmanh1312/durian-processed-data/data-leaf/train (3104 images)
Val:   /kaggle/input/datasets/tranmanh131

In [3]:
# BƯỚC 3: IMPORT THƯ VIỆN
import json
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tqdm import tqdm
import yaml
import cv2
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

2026-04-20 14:52:14.483283: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776696734.878042      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776696734.983909      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776696735.941539      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776696735.941586      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776696735.941589      23 computation_placer.cc:177] computation placer alr

All libraries imported successfully!


In [4]:
# BƯỚC 4: CẤU HÌNH MODEL
# Chọn model: resnet18, resnet34, resnet50, efficientnet_b0, mobilenetv3_large
MODEL_NAME = "mobilenetv3_large"  # THAY ĐỔI MODEL TẠI ĐÂY

MODEL_CONFIGS = {
    "resnet18": {
        "image_size": 224, "batch_size": 32, "num_epochs": 50, "lr": 0.0003, "dropout": 0.3,
    },
    "resnet34": {
        "image_size": 224, "batch_size": 28, "num_epochs": 60, "lr": 0.00025, "dropout": 0.35,
    },
    "resnet50": {
        "image_size": 256, "batch_size": 24, "num_epochs": 65, "lr": 0.0002, "dropout": 0.4, "mixed_precision": True,
    },
    "efficientnet_b0": {
        "image_size": 224, "batch_size": 32, "num_epochs": 50, "lr": 0.0001, "dropout": 0.3,
    },
    "mobilenetv3_large": {
        "image_size": 224, "batch_size": 48, "num_epochs": 50, "lr": 0.0001, "dropout": 0.2,
    },
}

cfg = MODEL_CONFIGS[MODEL_NAME]
OUTPUT_DIR = Path(f"/kaggle/working/results/{MODEL_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"===== Training {MODEL_NAME.upper()} =====")
print(f"Image size: {cfg['image_size']}, Batch size: {cfg['batch_size']}, Epochs: {cfg['num_epochs']}, LR: {cfg['lr']}")
print(f"Output: {OUTPUT_DIR}")

===== Training MOBILENETV3_LARGE =====
Image size: 224, Batch size: 48, Epochs: 50, LR: 0.0001
Output: /kaggle/working/results/mobilenetv3_large


In [5]:
# BƯỚC 5: THIẾT LẬP SEED & DEVICE
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # PyTorch >= 2.0 dùng total_memory, phiên bản cũ dùng total_mem
    vram = getattr(torch.cuda.get_device_properties(0), 'total_memory', None)
    if vram is None:
        vram = torch.cuda.get_device_properties(0).total_mem
    print(f"VRAM: {vram / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [6]:
# BƯỚC 6: DATASET & DATALOADER
CLASS_NAMES = ["ALGAL_LEAF_SPOT", "ALLOCARIDARA_ATTACK", "HEALTHY_LEAF", "LEAF_BLIGHT", "PHOMOPSIS_LEAF_SPOT"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")
IMAGE_SIZE = cfg["image_size"]
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

class DurianLeafDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, class_names, transforms=None, return_paths=False):
        self.root_dir = Path(root_dir)
        self.class_names = list(class_names)
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}
        self.transforms = transforms
        self.return_paths = return_paths
        self.samples = self._gather_samples()

    def _gather_samples(self):
        samples = []
        for cls in self.class_names:
            class_dir = self.root_dir / cls
            if not class_dir.exists():
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in IMG_EXTENSIONS:
                    samples.append((str(img_path), self.class_to_idx[cls]))
        samples.sort(key=lambda x: x[0])
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = np.array(Image.open(image_path).convert("RGB"))
        if self.transforms:
            transformed = self.transforms(image=image)
            image_tensor = transformed["image"]
        else:
            image_tensor = image
        sample = {"image": image_tensor, "label": label}
        if self.return_paths:
            sample["path"] = image_path
        return sample

train_transform = A.Compose([
    A.LongestMaxSize(IMAGE_SIZE),
    A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Rotate(limit=30, p=0.8),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), p=0.15),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.LongestMaxSize(IMAGE_SIZE),
    A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

train_dataset = DurianLeafDataset(DATA_ROOT / "train", CLASS_NAMES, train_transform)
val_dataset = DurianLeafDataset(DATA_ROOT / "val", CLASS_NAMES, eval_transform, return_paths=True)
test_dataset = DurianLeafDataset(DATA_ROOT / "test", CLASS_NAMES, eval_transform, return_paths=True)

from collections import Counter
labels = [l for _, l in train_dataset.samples]
counts = Counter(labels)
total = sum(counts.values())
weights = {cls: total / (NUM_CLASSES * cnt) for cls, cnt in counts.items()}
sample_weights = [weights[l] for l in labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, sampler=sampler, batch_size=cfg["batch_size"], num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=cfg["batch_size"], num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=cfg["batch_size"], num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"Val: {len(val_dataset)} images, Test: {len(test_dataset)} images")
print(f"Class distribution (train): {dict(counts)}")

Train: 3104 images (65 batches)
Val: 443 images, Test: 890 images
Class distribution (train): {0: 513, 1: 639, 2: 683, 3: 655, 4: 614}


In [7]:
# BƯỚC 7: BUILD MODEL
import torchvision.models as models

def create_model(name, num_classes, dropout):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    elif name == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.classifier[1].in_features, num_classes))
    elif name == "mobilenetv3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        # MobileNetV3-Large pretrained: avgpool output = 960 features
        # Original classifier[0] = Linear(960, 1280), classifier[3] = Linear(1280, 1000)
        # Replace ENTIRE classifier: Linear(960, 1280) → Hardswish → Dropout → Linear(1280, num_classes)
        in_features = m.classifier[0].in_features   # = 960
        hidden = m.classifier[0].out_features       # = 1280
        m.classifier = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout, inplace=True),
            nn.Linear(hidden, num_classes),
        )
    return m.to(DEVICE)

model = create_model(MODEL_NAME, NUM_CLASSES, cfg["dropout"])
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}, Total params: {total_params:,}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 146MB/s] 


Model: mobilenetv3_large, Total params: 4,618,037


In [8]:
# BƯỚC 8: TRAINING SETUP
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)

total_epochs = cfg["num_epochs"]
warmup_epochs = 3

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * min(progress, 1.0)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
use_amp = cfg.get("mixed_precision", False) and torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)
print(f"Mixed precision: {use_amp}")

Mixed precision: False


In [9]:
# BƯỚC 9: TRAINING LOOP
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
writer = SummaryWriter(OUTPUT_DIR / "tensorboard")

for epoch in range(1, total_epochs + 1):
    model.train()
    running_loss, running_acc = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{total_epochs} [Train]")
    for batch in pbar:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        optimizer.zero_grad()

        with autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = outputs.argmax(dim=1)
        acc = (preds == labels).float().mean().item()
        running_loss += loss.item()
        running_acc += acc
        pbar.set_postfix(loss=running_loss / (pbar.n + 1), acc=running_acc / (pbar.n + 1))

    train_loss = running_loss / len(train_loader)
    train_acc = running_acc / len(train_loader)

    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{total_epochs} [Val]", leave=False):
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            acc = (preds == labels).float().mean().item()
            val_loss += loss.item()
            val_acc += acc

    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": train_acc, "val": val_acc}, epoch)
    writer.add_scalar("LR", current_lr, epoch)

    print(f"Epoch {epoch:3d}/{total_epochs} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | LR: {current_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "best_metric": val_loss, "history": history}, OUTPUT_DIR / "best_model.pth")
        print(f"  [Saved] val_loss: {val_loss:.4f}")

writer.close()

with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print(f"\n===== Training Complete =====")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Results saved to: {OUTPUT_DIR}")

Epoch 1/50 [Train]:   0%|          | 0/65 [00:02<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (48x960 and 1280x1280)

In [ ]:
# BƯỚC 10: EVALUATE TRÊN TEST SET
import os
print(os.path.exists(OUTPUT_DIR / "best_model.pth"))
print(os.path.getsize(OUTPUT_DIR / "best_model.pth") if os.path.exists(OUTPUT_DIR / "best_model.pth") else "File not found")
checkpoint = torch.load(OUTPUT_DIR / "best_model.pth", map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating on test set"):
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4)

print(f"\n===== Test Set Results =====")
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

torch.save({"test_accuracy": accuracy, "confusion_matrix": cm, "classification_report": report, "all_preds": all_preds, "all_labels": all_labels}, OUTPUT_DIR / "evaluation_results.pt")

In [ ]:
# BƯỚC 11: VẼ CONFUSION MATRIX
# Load kết quả từ Cell 10 nếu chưa có, hoặc từ file
try:
    # Thử lấy từ biến đã tính ở Cell 10
    cm_local = cm
    acc_local = accuracy
except NameError:
    # Fallback: load từ file kết quả đã lưu
    results = torch.load(OUTPUT_DIR / "evaluation_results.pt", weights_only=False)
    cm_local = results["confusion_matrix"]
    acc_local = results["test_accuracy"]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_local, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"{MODEL_NAME.upper()} — Confusion Matrix (Accuracy: {acc_local:.4f})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.close()
print(f"Confusion matrix saved!")

In [ ]:
# BƯỚC 12: VẼ TRAINING CURVES
# Load từ file nếu chưa có biến history (chạy cell riêng lẻ)
try:
    hist = history
except NameError:
    with open(OUTPUT_DIR / "training_history.json", "r") as f:
        hist = json.load(f)
    hist = {k: [float(v) for v in vals] for k, vals in hist.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist["train_loss"], label="Train Loss")
axes[0].plot(hist["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(hist["train_acc"], label="Train Acc")
axes[1].plot(hist["val_acc"], label="Val Acc")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].legend()
axes[1].grid(True)

plt.suptitle(f"{MODEL_NAME.upper()} — Training Curves")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.close()
print(f"Training curves saved!")

In [ ]:
# Nén file best_model.pth thành .zip để tải về dễ dàng hơn
import zipfile, os

zip_path = OUTPUT_DIR / "best_model.zip"
if not zip_path.exists():
    print("Compressing best_model.pth...")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_DIR / "best_model.pth", arcname="best_model.pth")
    # Nén thêm history
    with zipfile.ZipFile(zip_path, "a", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_DIR / "training_history.json", arcname="training_history.json")
    print(f"Done! Size: {zip_path.stat().st_size / 1e6:.1f} MB")

from IPython.display import FileLink, display
display(FileLink(zip_path))

In [ ]:
# BƯỚC 13: TẢI KẾT QUẢ VỀ MÁY
from IPython.display import FileLink, display

print("===== Download Links =====")
display(FileLink(OUTPUT_DIR / "best_model.pth"))
display(FileLink(OUTPUT_DIR / "training_history.json"))
display(FileLink(OUTPUT_DIR / "evaluation_results.pt"))
display(FileLink(OUTPUT_DIR / "confusion_matrix.png"))
display(FileLink(OUTPUT_DIR / "training_curves.png"))

In [ ]:
from IPython.display import FileLink

# Thay tên file bằng đường dẫn chính xác của bạn
FileLink(r'/kaggle/working/results/resnet18/best_model.pth') 
# Hoặc FileLink(r'best_model.zip')